# Bayesian Logistic Regression

## Overview

## Bayesian Logistic Regression

Bayesian Logistic Regression is applied to the customer repeat-purchase problem
because the target variable is binary:

- RepeatPurchase = 1 if the customer purchases again in the future
- RepeatPurchase = 0 if the customer does not purchase again

The same historical customer predictors used in the modelling notebook are used here so that
the Bayesian approach addresses the same business problem.

Unlike standard Logistic Regression, Bayesian Logistic Regression estimates
a probability distribution for each model coefficient. This allows uncertainty
about the effect of each customer behaviour to be quantified.

In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

In [ ]:
# ============================================================
# 2. LOAD MERCHANDISE TRANSACTION DATA
# ============================================================

from pathlib import Path

base_path = Path("../data/processed")

df_merchandise = pd.read_csv(
    base_path / "online_retail_merchandise.csv"
)

# Convert InvoiceDate so transactions can be separated by time.
df_merchandise["InvoiceDate"] = pd.to_datetime(
    df_merchandise["InvoiceDate"]
)

print("Dataset shape:", df_merchandise.shape)

print("\nDate range:")
print("Minimum:", df_merchandise["InvoiceDate"].min())
print("Maximum:", df_merchandise["InvoiceDate"].max())

In [ ]:
# ============================================================
# 3. DEFINE HISTORICAL AND FUTURE PERIODS
# ============================================================

# Historical period:
# Used to build customer behaviour predictors.

observation_start = pd.Timestamp("2010-06-10")
observation_end = pd.Timestamp("2011-06-09 23:59:59")

# Future period:
# Used only to determine whether the customer purchased again.

outcome_start = pd.Timestamp("2011-06-10")
outcome_end = pd.Timestamp("2011-12-09 23:59:59")

print("Historical period:")
print(observation_start, "to", observation_end)

print("\nFuture period:")
print(outcome_start, "to", outcome_end)

In [ ]:
# ============================================================
# 4. SPLIT INTO HISTORICAL AND FUTURE DATA
# ============================================================

df_history = df_merchandise[
    (df_merchandise["InvoiceDate"] >= observation_start) &
    (df_merchandise["InvoiceDate"] <= observation_end)
].copy()

df_future = df_merchandise[
    (df_merchandise["InvoiceDate"] >= outcome_start) &
    (df_merchandise["InvoiceDate"] <= outcome_end)
].copy()

print("Historical rows:", len(df_history))
print("Future rows:", len(df_future))

In [ ]:
# ============================================================
# 5. CREATE FUTURE REPEAT-PURCHASE TARGET
# ============================================================

# Only customers with historical behaviour are eligible.

historical_customers = (
    df_history["Customer ID"]
    .dropna()
    .unique()
)

# Customers who appear in the future period.

future_customers = set(
    df_future["Customer ID"]
    .dropna()
    .unique()
)

df_model = pd.DataFrame({
    "CustomerID": historical_customers
})

df_model["RepeatPurchase"] = (
    df_model["CustomerID"]
    .isin(future_customers)
    .astype(int)
)

print("Eligible customers:", len(df_model))

print("\nTarget counts:")
print(df_model["RepeatPurchase"].value_counts())

print("\nTarget percentages:")
print(
    df_model["RepeatPurchase"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# ============================================================
# 6. CREATE HISTORICAL ORDER-LEVEL DATA
# ============================================================

# The merchandise dataset contains one row per product line.
# We first combine product lines into one row per invoice/order.

history_orders = (
    df_history
    .groupby("Invoice", as_index=False)
    .agg(
        CustomerID=("Customer ID", "first"),
        OrderDate=("InvoiceDate", "min"),
        OrderValue=("TransactionValue", "sum")
    )
)

print("Historical orders:", len(history_orders))
print("Historical customers:", history_orders["CustomerID"].nunique())

In [ ]:
# ============================================================
# 7. CREATE HISTORICAL CUSTOMER FEATURES
# ============================================================

historical_features = (
    history_orders
    .groupby("CustomerID")
    .agg(
        # Number of historical orders
        Frequency=("Invoice", "nunique"),

        # Total historical spending
        Monetary=("OrderValue", "sum"),

        # Average value per order
        AverageOrderValue=("OrderValue", "mean"),

        # First historical purchase
        FirstPurchaseDate=("OrderDate", "min"),

        # Most recent historical purchase
        LastPurchaseDate=("OrderDate", "max")
    )
    .reset_index()
)

In [ ]:
# ============================================================
# 8. CREATE RECENCY AND CUSTOMER TENURE
# ============================================================

reference_date = pd.Timestamp("2011-06-10")

# Recency = days since last historical purchase
historical_features["Recency"] = (
    reference_date -
    historical_features["LastPurchaseDate"]
).dt.days

# CustomerTenure = days between first and last historical purchase
historical_features["CustomerTenure"] = (
    historical_features["LastPurchaseDate"] -
    historical_features["FirstPurchaseDate"]
).dt.days

In [ ]:
# ============================================================
# 8. CREATE RECENCY AND CUSTOMER TENURE
# ============================================================

reference_date = pd.Timestamp("2011-06-10")

# Recency = days since last historical purchase
historical_features["Recency"] = (
    reference_date -
    historical_features["LastPurchaseDate"]
).dt.days

# CustomerTenure = days between first and last historical purchase
historical_features["CustomerTenure"] = (
    historical_features["LastPurchaseDate"] -
    historical_features["FirstPurchaseDate"]
).dt.days

In [ ]:
# ============================================================
# 9. CREATE PRODUCT DIVERSITY
# ============================================================

product_diversity = (
    df_history
    .groupby("Customer ID")["StockCode"]
    .nunique()
    .reset_index(name="ProductDiversity")
)

product_diversity = product_diversity.rename(
    columns={"Customer ID": "CustomerID"}
)

historical_features = historical_features.merge(
    product_diversity,
    on="CustomerID",
    how="left"
)

In [ ]:
# ============================================================
# 10. CREATE AVERAGE PURCHASE INTERVAL
# ============================================================

def calculate_purchase_interval(group):

    dates = group["OrderDate"].sort_values()

    intervals = dates.diff().dt.days.dropna()

    if len(intervals) == 0:
        return np.nan

    return intervals.mean()


purchase_interval = (
    history_orders
    .groupby("CustomerID")
    .apply(calculate_purchase_interval)
    .reset_index(name="AveragePurchaseInterval")
)

historical_features = historical_features.merge(
    purchase_interval,
    on="CustomerID",
    how="left"
)

In [ ]:
# ============================================================
# 11. MERGE HISTORICAL FEATURES WITH REPEAT-PURCHASE TARGET
# ============================================================

df_model = df_model.merge(
    historical_features,
    on="CustomerID",
    how="left"
)

print("Modelling dataset shape:", df_model.shape)

print("\nColumns:")
print(df_model.columns.tolist())

print("\nMissing values:")
print(df_model.isnull().sum())

In [ ]:
# ============================================================
# 12. HANDLE MISSING PURCHASE INTERVAL
# ============================================================

# Missing AveragePurchaseInterval usually means that the customer
# made only one historical purchase.
#
# Instead of deleting these customers, we:
# 1. create an indicator showing whether an interval exists
# 2. replace missing interval values with 0

df_model["HasPurchaseInterval"] = (
    df_model["AveragePurchaseInterval"]
    .notna()
    .astype(int)
)

df_model["AveragePurchaseInterval"] = (
    df_model["AveragePurchaseInterval"]
    .fillna(0)
)

print("Remaining missing values:")
print(df_model.isnull().sum())

In [ ]:
# ============================================================
# 13. LOG-TRANSFORM SKEWED PREDICTORS
# ============================================================

# These variables were highly right-skewed in Task 5.
# log1p(x) reduces the effect of extreme values.

df_model["Log_Frequency"] = np.log1p(
    df_model["Frequency"]
)

df_model["Log_AverageOrderValue"] = np.log1p(
    df_model["AverageOrderValue"]
)

df_model["Log_ProductDiversity"] = np.log1p(
    df_model["ProductDiversity"]
)

df_model["Log_AveragePurchaseInterval"] = np.log1p(
    df_model["AveragePurchaseInterval"]
)

In [ ]:
# ============================================================
# 14. DEFINE FINAL BAYESIAN PREDICTORS
# ============================================================

final_predictors = [
    "Log_Frequency",
    "Recency",
    "Log_AverageOrderValue",
    "Log_ProductDiversity",
    "CustomerTenure",
    "Log_AveragePurchaseInterval",
    "HasPurchaseInterval"
]

X = df_model[final_predictors].copy()
y = df_model["RepeatPurchase"].copy()

print("Predictors:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nMissing values:")
print(X.isnull().sum())

In [ ]:
# ============================================================
# 15. TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training customers:", X_train.shape[0])
print("Testing customers:", X_test.shape[0])

In [ ]:
# ============================================================
# 16. STANDARDIZE PREDICTORS
# ============================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit using training data only to avoid leakage.
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to test data.
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

In [ ]:
# ============================================================
# 17. IMPORT BAYESIAN MODELLING LIBRARIES
# ============================================================

# PyMC is used to build the Bayesian Logistic Regression model.
# ArviZ is used to summarize posterior distributions.

import pymc as pm
import arviz as az
import numpy as np

print("PyMC version:", pm.__version__)

In [ ]:
# ============================================================
# 18. PREPARE DATA FOR BAYESIAN LOGISTIC REGRESSION
# ============================================================

# Convert the scaled training data into numpy arrays.
# PyMC works cleanly with numerical arrays.

X_train_bayes = np.asarray(
    X_train_scaled,
    dtype=float
)

y_train_bayes = np.asarray(
    y_train,
    dtype=int
)

print("Bayesian X shape:", X_train_bayes.shape)
print("Bayesian y shape:", y_train_bayes.shape)

In [ ]:
# ============================================================
# 19. BUILD BAYESIAN LOGISTIC REGRESSION MODEL
# ============================================================

# The target is binary:
# RepeatPurchase = 1 or 0
#
# Therefore, we use a Bernoulli likelihood with a logistic link.

with pm.Model() as bayesian_logistic_model:

    # --------------------------------------------------------
    # INTERCEPT PRIOR
    # --------------------------------------------------------
    #
    # The intercept represents the baseline log-odds of repeat purchase.
    #
    # A Normal prior centred at 0 is used.
    # sigma=2 is broad enough to allow different plausible values.

    intercept = pm.Normal(
        "intercept",
        mu=0,
        sigma=2
    )

    # --------------------------------------------------------
    # COEFFICIENT PRIORS
    # --------------------------------------------------------
    #
    # Each predictor gets its own coefficient.
    #
    # Because the predictors were standardized, a Normal(0,1)
    # prior is a reasonable weakly informative prior.
    #
    # The prior says large positive or negative effects are possible,
    # but extremely large coefficients are less likely.

    beta = pm.Normal(
        "beta",
        mu=0,
        sigma=1,
        shape=X_train_bayes.shape[1]
    )

    # --------------------------------------------------------
    # LINEAR PREDICTOR
    # --------------------------------------------------------
    #
    # This is similar to ordinary Logistic Regression:
    #
    # intercept + beta1*x1 + beta2*x2 + ...

    logit_p = (
        intercept
        + pm.math.dot(X_train_bayes, beta)
    )

    # Convert log-odds into probability between 0 and 1.

    p = pm.Deterministic(
        "p",
        pm.math.sigmoid(logit_p)
    )

    # --------------------------------------------------------
    # BINARY OUTCOME
    # --------------------------------------------------------
    #
    # RepeatPurchase follows a Bernoulli distribution because
    # each customer either purchases again (1) or does not (0).

    repeat_purchase = pm.Bernoulli(
        "RepeatPurchase",
        p=p,
        observed=y_train_bayes
    )

    # --------------------------------------------------------
    # POSTERIOR SAMPLING
    # --------------------------------------------------------
    #
    # The sampler combines the prior distributions and observed data
    # to estimate posterior distributions for the coefficients.

    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=2,
        random_seed=42,
        target_accept=0.9
    )

In [ ]:
# ============================================================
# 21. MAP BAYESIAN COEFFICIENTS TO PREDICTOR NAMES
# ============================================================

# Extract posterior mean for each predictor coefficient.

posterior_beta_means = (
    trace.posterior["beta"]
    .mean(dim=["chain", "draw"])
    .values
)

bayesian_coef_table = pd.DataFrame({
    "Predictor": final_predictors,
    "PosteriorMean": posterior_beta_means
})

# Convert coefficients to odds ratios for easier comparison
# with the standard Logistic Regression model.

bayesian_coef_table["OddsRatio"] = np.exp(
    bayesian_coef_table["PosteriorMean"]
)

print(
    bayesian_coef_table
    .sort_values(
        "PosteriorMean",
        key=abs,
        ascending=False
    )
    .round(4)
)

In [ ]:
# ============================================================
# 22. SHOW 95% POSTERIOR INTERVALS
# ============================================================

bayes_summary = az.summary(
    trace,
    var_names=["beta"],
    hdi_prob=0.95
)

print(bayes_summary)

In [ ]:
# ============================================================
# 23. BAYESIAN COEFFICIENT FOREST PLOT WITH 95% HDIs
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Business-friendly predictor names
display_names = [
    "Purchase Frequency",
    "Recency",
    "Average Order Value",
    "Product Diversity",
    "Customer Tenure",
    "Average Purchase Interval",
    "Has Purchase Interval"
]

# Extract posterior means and 95% HDI limits
plot_df = pd.DataFrame({
    "Predictor": display_names,
    "Mean": bayes_summary["mean"].values,
    "Lower": bayes_summary["hdi_2.5%"].values,
    "Upper": bayes_summary["hdi_97.5%"].values
})

# Sort for a cleaner visual
plot_df = plot_df.sort_values("Mean")

# Calculate asymmetric error-bar distances
lower_error = plot_df["Mean"] - plot_df["Lower"]
upper_error = plot_df["Upper"] - plot_df["Mean"]

plt.figure(figsize=(8, 5))

plt.errorbar(
    plot_df["Mean"],
    plot_df["Predictor"],
    xerr=[lower_error, upper_error],
    fmt="o",
    capsize=4
)

# Zero = no coefficient effect
plt.axvline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel("Posterior Coefficient")
plt.ylabel("")

plt.title(
    "Bayesian Logistic Regression Coefficients\nwith 95% HDIs",
    fontweight="bold"
)

plt.grid(axis="x", alpha=0.20)

plt.tight_layout()
plt.show()